In [1]:
# ── Cell 1: Imports ────────────────────────────────────────────────────────────
import glob
import io
import os
import shutil
import sys
import time
import pathlib
import re
from collections import OrderedDict
from datetime import datetime, timedelta
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import polars as pl
import pyautogui
import win32clipboard

from PIL import Image, ImageGrab

from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import (
    NoAlertPresentException,
    NoSuchElementException,
    TimeoutException,
    StaleElementReferenceException
)

from IPython.display import HTML

In [2]:
# ── Cell 2: Helper — read all xlsx/csv files from a folder into one DataFrame ──
def input_data(data_dir):
    list_files = []
    for path in pathlib.Path(data_dir).glob('**/*.*'):
        if path.suffix.lower() in ['.xlsx', '.csv'] and path.stat().st_size > 0:
            export_time = datetime.fromtimestamp(path.stat().st_mtime)
            try:
                df = (
                    pl.read_excel(path)
                    if path.suffix.lower() == '.xlsx'
                    else pl.read_csv(path, infer_schema_length=10000, ignore_errors=True)
                )
                if not df.is_empty():
                    df = df.with_columns([
                        pl.lit(path.stem).alias('sheet_name'),
                        pl.lit(export_time).alias('Export time')
                    ])
                    list_files.append(df)
            except Exception:
                # Skip unreadable files silently
                continue

    return pl.concat(list_files, how="diagonal_relaxed") if list_files else pl.DataFrame()

In [3]:
# ── Cell 3: Path config & load HC reference table ──────────────────────────────
first_glob = os.path.expanduser("~").replace("//", "/")

folder_paths = {
    "req"            : f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OU.xlsx",
    "current_agents" : f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/CAPTURE/current_agent/",
    "input_iex"      : f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/CAPTURE/current_iex/",
}

# Load headcount reference: used later to enrich IEX data with agent info
hc_extend_combination = (
    pl.read_parquet(
        f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/hc_extend_combination.parquet"
    )
    .select(["Date", "IEX ID", "Employee Name", "LOB", "Email Id", "Supervisor Name"])
    .with_columns([
        pl.col("Date").cast(pl.Date, strict=False),
        # Strip stray quotes before casting IEX ID to integer
        pl.col("IEX ID").str.replace_all('"', "").cast(pl.Int64, strict=False)
    ])
    .filter(pl.col("Date") >= pl.date(2026, 1, 1))
)

hc_extend_combination

Date,IEX ID,Employee Name,LOB,Email Id,Supervisor Name
date,i64,str,str,str,str
2026-01-01,3061192,"""TO ANH DUY""","""Support""","""anhduy.to@concentrix.com""","""Patar Kirpan"""
2026-01-01,3093976,"""NGUYEN QUANG DUY""","""Lodging""","""quangduy.nguyen2@concentrix.co…","""Tran Sharon"""
2026-01-01,3109754,"""LE HOAI MINH NGAN""","""Lodging""","""hoaiminhngan.le@concentrix.com""","""Chau Thien Kim"""
2026-01-01,3112792,"""TRAN VU HOAI LINH""","""Lodging""","""vuhoailinh.tran@concentrix.com""","""Truong Thien Thanh Toan"""
2026-01-01,3112804,"""NGUYEN THI THU THUY""","""Lodging""","""thithuthuy.nguyen@concentrix.c…","""Truong Thien Thanh Toan"""
…,…,…,…,…,…
2026-06-26,3109420,"""NGUYEN BUI THANH TRUC""","""Lodging""","""buithanhtruc.nguyen@concentrix…","""Truong Thien Thanh Toan"""
2026-06-26,3002196,"""PHUNG MY UYEN""","""Lodging""","""myuyen.phung@concentrix.com""","""Truong Thien Thanh Toan"""
2026-06-26,null,"""LE THI THU MINH""","""Support - NL""","""thithuminh.le@concentrix.com""","""Patar Kirpan"""


In [4]:
# ── Cell 4: Parse OU (Occupancy/Requirement) file ──────────────────────────────
def process_ou_hours(file_path):
    try:
        # Read raw without header so we can handle datetime column names
        df_raw = pl.read_excel(source=file_path, has_header=False, infer_schema_length=0)

        # Convert first row (header) — format dates as YYYY-MM-DD strings
        header_vals = df_raw.row(0)
        new_columns = [
            val.strftime("%Y-%m-%d") if isinstance(val, datetime)
            else str(val).strip() if val is not None
            else "Unknown"
            for val in header_vals
        ]
        df_raw.columns = new_columns

        # Drop the header row and clean the Interval column
        df = df_raw.slice(1).with_columns(
            pl.col("Interval")
              .cast(pl.String)
              .str.replace(r"^.*1899-12-31\s+", "")
              .str.slice(0, 8)
              .alias("Interval")
        )

        final_df = (
            df.unpivot(
                index=["LOB", "Site", "Interval"],
                variable_name="Date_Str",
                value_name="Value"
            )
            .with_columns([
                # Build PST datetime from date string + interval time
                (pl.col("Date_Str").str.slice(0, 10) + " " + pl.col("Interval"))
                  .str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False)
                  .dt.truncate("1m")
                  .alias("PST_Datetime"),

                pl.col("Value").cast(pl.Float64, strict=False).fill_null(0.0).alias("OU Req by Site"),
                pl.col("LOB").str.strip_chars(),
                pl.col("Site").str.strip_chars()
            ])
            # Attach PST timezone before converting to avoid DST ambiguity
            .with_columns(
                pl.col("PST_Datetime")
                  .dt.replace_time_zone("America/Los_Angeles", non_existent="null", ambiguous="earliest")
                  .alias("_tz_aware")
            )
            .with_columns([
                pl.col("_tz_aware").dt.convert_time_zone("Asia/Ho_Chi_Minh").dt.replace_time_zone(None).alias("VNT_Datetime"),
                pl.col("_tz_aware").dt.convert_time_zone("Africa/Cairo").dt.replace_time_zone(None).alias("CLT_Datetime"),
                pl.col("_tz_aware").dt.convert_time_zone("Asia/Kolkata").dt.replace_time_zone(None).alias("IST_Datetime"),
                # Total OU = sum across all sites for the same LOB+interval
                pl.col("OU Req by Site").sum().over(["LOB", "PST_Datetime"]).alias("Total OU Req")
            ])
            .drop("_tz_aware")
        )

        return final_df.select([
            "LOB", "Site", "PST_Datetime",
            "VNT_Datetime", "CLT_Datetime", "IST_Datetime",
            "Total OU Req", "OU Req by Site"
        ])

    except Exception as e:
        print(f"Error processing file: {e}")
        return pl.DataFrame()

df_req = process_ou_hours(folder_paths["req"])

In [5]:
# ── Cell 5: Constants, outage DB, headcount helpers ────────────────────────────

# Mapping from full site name (in OU file) to short code
REQ_SITE_MAPPING = {
    "Concentrix (Pune)"          : "PUN",
    "Concentrix (Ho Chi Minh City)": "HCM",
    "Concentrix (Kolkata)"       : "KOL",
    "Concentrix (Cairo)"         : "CAI"
}

# Mapping from partial site name (in agent export) to short code
SITE_MAPPING = {"Pune": "PUN", "Ho Chi Minh": "HCM", "Kolkata": "KOL", "Cairo": "CAI"}

# Which routing profiles belong to each LOB
LOB_MAPPING = {
    "Lodging Chat"    : ["Chat_OD_EN_Car_Activity", "Chat_OD_EN_Lodging",
                         "Chat - Global English Lodging Nesting", "Chat_Lodging English w Car"],
    "Non Lodging Chat": ["Chat - Global English Non- Lodging Nesting",
                         "Chat_OD_EN_Dual_GDS", "Chat_OD_EN_Amadeus"]
}

# Connect states considered "active" (handling a customer)
ACTIVE_STATES = ["AVAILABLECHAT", "OUTBOUNDCALL"]

# ── Build outage_db: one row per agent per 30-min interval ────────────────────
outage_db = (
    input_data(folder_paths["current_agents"])
    # Keep only the latest 16 export snapshots (8 hours worth of 30-min intervals)
    .filter(
        pl.col("Export time").is_in(
            input_data(folder_paths["current_agents"])
            .select("Export time").unique()
            .sort("Export time", descending=True)
            .limit(16)
            .get_column("Export time")
        )
    )
    .select([
        "Business Location", "Export time", "Agent Name", "Agent Email",
        "Agent Manager", "Connect State", "Assigned Workitem Count",
        "Duration", "Queue Group / Routing Profile"
    ])
    .with_columns([
        # Truncate export timestamp to 30-min bucket (VNT)
        pl.col("Export time").cast(pl.Datetime, strict=False).dt.truncate("30m").alias("VNT_Datetime"),
        pl.col("Duration").str.strptime(pl.Time, "%H:%M:%S", strict=False)
    ])
    # Within each 30-min bucket keep only the latest snapshot per agent
    .filter(pl.col("Export time") == pl.col("Export time").max().over("VNT_Datetime"))
    .with_columns([
        pl.col("VNT_Datetime")
          .dt.replace_time_zone("Asia/Ho_Chi_Minh")
          .dt.convert_time_zone("America/Los_Angeles")
          .dt.replace_time_zone(None)
          .alias("PST_Datetime")
    ])
    # Map routing profile → LOB label
    .with_columns([
        pl.when(pl.col("Queue Group / Routing Profile").is_in(LOB_MAPPING["Lodging Chat"]))
          .then(pl.lit("Lodging Chat"))
          .when(pl.col("Queue Group / Routing Profile").is_in(LOB_MAPPING["Non Lodging Chat"]))
          .then(pl.lit("Non Lodging Chat"))
          .otherwise(None)
          .alias("LOB")
    ])
    .filter(pl.col("LOB").is_not_null())
)

# Boolean conditions for active / inactive agents
active_cond = (
    pl.col("Connect State").is_in(ACTIVE_STATES) |
    (pl.col("Assigned Workitem Count").cast(pl.Int32, strict=False).fill_null(0) > 0)
)
inactive_cond = ~active_cond


# ── Helper: add multi-timezone time columns from a PST datetime column ─────────
def add_time_cols(df, col="PST_Datetime"):
    return (
        df.with_columns(
            pl.col(col)
              .dt.replace_time_zone("America/Los_Angeles", non_existent="null", ambiguous="earliest")
              .alias("_tz")
        )
        .with_columns([
            pl.col(col).dt.date().alias("PST_Date"),
            pl.col(col).dt.strftime("%H:%M").alias("PST"),
            pl.col("_tz").dt.convert_time_zone("Asia/Kolkata").dt.replace_time_zone(None).dt.strftime("%H:%M").alias("IST"),
            pl.col("_tz").dt.convert_time_zone("Africa/Cairo").dt.replace_time_zone(None).dt.strftime("%H:%M").alias("CLT"),
            pl.col("_tz").dt.convert_time_zone("Asia/Ho_Chi_Minh").dt.replace_time_zone(None).dt.strftime("%H:%M").alias("VNT")
        ])
        .drop("_tz")
    )


# ── Helper: count unique agents per site/LOB/interval for a given condition ────
def build_hc(df, cond, label):
    value_col = f"Heads {label}"
    return (
        df.group_by(["LOB", "PST_Datetime"])
          .agg([
              pl.col("Agent Name")
                .filter(pl.col("Business Location").str.contains(loc) & cond)
                .n_unique()
                .alias(prefix)
              for loc, prefix in SITE_MAPPING.items()
          ])
          .fill_null(0)
          .sort(["LOB", "PST_Datetime"])
          .unpivot(
              index=["LOB", "PST_Datetime"],
              on=list(SITE_MAPPING.values()),
              variable_name="Site",
              value_name=value_col
          )
          .pipe(add_time_cols)
          .select(["LOB", "Site", value_col, "PST_Datetime", "PST_Date", "PST", "IST", "CLT", "VNT"])
          .sort(["LOB", "Site"])
    )

df_active  = build_hc(outage_db, active_cond,   "Active")
df_inactive = build_hc(outage_db, inactive_cond, "Inactive")

# ── Normalise OU requirement data to match agent data column names ─────────────
df_req = (
    df_req
    .with_columns([
        pl.col("Site").replace(REQ_SITE_MAPPING),
        # Normalise LOB labels to match outage_db
        pl.when(pl.col("LOB") == "Lodging chat").then(pl.lit("Lodging Chat"))
          .when(pl.col("LOB") == "Non-Lodging chat").then(pl.lit("Non Lodging Chat"))
          .otherwise(pl.col("LOB"))
          .alias("LOB"),
        # OU file stores 30-min values; multiply by 2 to get hourly headcount
        (pl.col("Total OU Req") * 2).alias("Total OU Req"),
        (pl.col("OU Req by Site") * 2).alias("OU Req by Site")
    ])
    .select(["LOB", "Site", "PST_Datetime", "Total OU Req", "OU Req by Site"])
)

# ── Join OU requirements into active headcount table ──────────────────────────
df_active = (
    df_active
    .join(df_req, on=["LOB", "Site", "PST_Datetime"], how="left")
    .with_columns([
        (pl.col("Heads Active") - pl.col("OU Req by Site")).round(1).alias("Missing Heads"),
        # Show "-" when no requirement is set for that site/interval
        pl.when(
            pl.col("OU Req by Site").is_null() | (pl.col("OU Req by Site") == 0)
        )
          .then(pl.lit("-"))
          .otherwise(
              ((pl.col("Heads Active") / pl.col("OU Req by Site") * 100)
               .round(1).cast(pl.String) + "%")
          )
          .alias("Heads Active / OU Req by Site"),
        pl.col("Total OU Req").round(1),
        pl.col("OU Req by Site").round(1)
    ])
    .select([
        "LOB", "Site", "Total OU Req", "OU Req by Site",
        "Heads Active", "Missing Heads", "Heads Active / OU Req by Site",
        "PST_Datetime", "PST_Date", "PST", "IST", "CLT", "VNT"
    ])
)

# ── Inactive agent detail (all intervals) ─────────────────────────────────────
df_inactive_details = (
    outage_db
    .filter(inactive_cond)
    .select(["Agent Name", "Agent Email", "LOB", "Business Location",
             "Connect State", "Duration", "PST_Datetime"])
    .pipe(add_time_cols)
)

# ── Snapshot: latest available interval ───────────────────────────────────────
latest_dt = outage_db.select(pl.col("PST_Datetime").max()).item()

df_active           = df_active.filter(pl.col("PST_Datetime") == latest_dt)
df_inactive         = df_inactive.filter(pl.col("PST_Datetime") == latest_dt)
df_inactive_details = df_inactive_details.filter(pl.col("PST_Datetime") == latest_dt)

# Split inactive details by LOB for separate display
df_lodging     = df_inactive_details.filter(pl.col("LOB") == "Lodging Chat").sort(["Business Location", "Agent Name"])
df_non_lodging = df_inactive_details.filter(pl.col("LOB") == "Non Lodging Chat").sort(["Business Location", "Agent Name"])

with pl.Config(tbl_rows=-1):
    display(df_inactive_details)
    display(df_lodging)
    display(df_non_lodging)
    display(df_active)
    display(df_inactive)

Agent Name,Agent Email,LOB,Business Location,Connect State,Duration,PST_Datetime,PST_Date,PST,IST,CLT,VNT
str,str,str,str,str,time,datetime[μs],date,str,str,str,str
"""Bui, Ba Duong""","""baduong.bui@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""TRAINING""",05:26:51,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Bui, Cong Danh""","""congdanh.bui@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:16:54,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""DAS, MOUMITA""","""moumita.das5@concentrix.com""","""Non Lodging Chat""","""Concentrix (Kolkata)""","""LOGIN""",02:11:17,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Hassan, Zaid""","""zaid.hassan@concentrix.com""","""Non Lodging Chat""","""Concentrix (Kolkata)""","""ENDOFSHIFT""",06:56:23,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Hoang, Le Vy""","""levy.hoang@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:17:10,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Khaled Saeed Elbadawy, Hassan""","""hassankhaled.elbadawy@concentr…","""Non Lodging Chat""","""Concentrix (Cairo)""","""ENDOFSHIFT""",13:27:09,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Lai, Ngoc Ai Linh""","""ngocailinh.lai@concentrix.com""","""Non Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:05:17,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Le, MINH QUAN""","""minhquan.le@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""TRAINING""",05:20:57,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""MAHTO, TEJASWI""","""tejaswi.mahto@concentrix.com""","""Lodging Chat""","""Concentrix (Pune)""","""LOGIN""",08:26:22,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""


Agent Name,Agent Email,LOB,Business Location,Connect State,Duration,PST_Datetime,PST_Date,PST,IST,CLT,VNT
str,str,str,str,str,time,datetime[μs],date,str,str,str,str
"""Bui, Ba Duong""","""baduong.bui@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""TRAINING""",05:26:51,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Bui, Cong Danh""","""congdanh.bui@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:16:54,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Hoang, Le Vy""","""levy.hoang@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:17:10,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Le, MINH QUAN""","""minhquan.le@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""TRAINING""",05:20:57,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Nguyen, Quang Hien""","""quanghien.nguyen@concentrix.co…","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:02:09,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Nguyen, Thi Kim Ngan""","""thikimngan.nguyen11@concentrix…","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:18:49,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Nguyen, Thi Thien An""","""thithienan.nguyen@concentrix.c…","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""TRAINING""",05:30:17,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Pham, Thanh Vu""","""thanhvu.pham@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:32:37,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Pham, Thi Tuyet Nhi""","""thituyetnhi.pham2@concentrix.c…","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LOGIN""",18:51:41,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""


Agent Name,Agent Email,LOB,Business Location,Connect State,Duration,PST_Datetime,PST_Date,PST,IST,CLT,VNT
str,str,str,str,str,time,datetime[μs],date,str,str,str,str
"""Khaled Saeed Elbadawy, Hassan""","""hassankhaled.elbadawy@concentr…","""Non Lodging Chat""","""Concentrix (Cairo)""","""ENDOFSHIFT""",13:27:09,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Refaat Ahmed, Youssef""","""youssef.refaatahmed@concentrix…","""Non Lodging Chat""","""Concentrix (Cairo)""","""ENDOFSHIFT""",04:25:28,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Sayed Fawzy Saad, Muhammed""","""muhammed.fawzysaad@concentrix.…","""Non Lodging Chat""","""Concentrix (Cairo)""","""LOGIN""",21:39:02,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Lai, Ngoc Ai Linh""","""ngocailinh.lai@concentrix.com""","""Non Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:05:17,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Trinh, Hoang Nam""","""hoangnam.trinh@concentrix.com""","""Non Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:27:29,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""DAS, MOUMITA""","""moumita.das5@concentrix.com""","""Non Lodging Chat""","""Concentrix (Kolkata)""","""LOGIN""",02:11:17,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Hassan, Zaid""","""zaid.hassan@concentrix.com""","""Non Lodging Chat""","""Concentrix (Kolkata)""","""ENDOFSHIFT""",06:56:23,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""PAUL, DIPANJAN""","""dipanjan.paul@concentrix.com""","""Non Lodging Chat""","""Concentrix (Kolkata)""","""ENDOFSHIFT""",06:02:59,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""SAHU, PROGATI""","""progati.sahu@concentrix.com""","""Non Lodging Chat""","""Concentrix (Kolkata)""","""LUNCH""",00:24:39,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""


LOB,Site,Total OU Req,OU Req by Site,Heads Active,Missing Heads,Heads Active / OU Req by Site,PST_Datetime,PST_Date,PST,IST,CLT,VNT
str,str,f64,f64,u32,f64,str,datetime[μs],date,str,str,str,str
"""Lodging Chat""","""CAI""",null,null,0,null,"""-""",2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Lodging Chat""","""HCM""",25.0,25.0,13,-12.0,"""52.0%""",2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Lodging Chat""","""KOL""",25.0,0.0,0,0.0,"""-""",2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Lodging Chat""","""PUN""",25.0,0.0,0,0.0,"""-""",2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Non Lodging Chat""","""CAI""",20.6,10.1,9,-1.1,"""88.9%""",2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Non Lodging Chat""","""HCM""",20.6,3.4,4,0.6,"""116.6%""",2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Non Lodging Chat""","""KOL""",20.6,7.0,6,-1.0,"""85.6%""",2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Non Lodging Chat""","""PUN""",20.6,0.0,0,0.0,"""-""",2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""


LOB,Site,Heads Inactive,PST_Datetime,PST_Date,PST,IST,CLT,VNT
str,str,u32,datetime[μs],date,str,str,str,str
"""Lodging Chat""","""CAI""",0,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Lodging Chat""","""HCM""",10,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Lodging Chat""","""KOL""",1,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Lodging Chat""","""PUN""",1,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Non Lodging Chat""","""CAI""",3,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Non Lodging Chat""","""HCM""",2,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Non Lodging Chat""","""KOL""",4,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Non Lodging Chat""","""PUN""",0,2026-05-13 21:00:00,2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00"""


In [6]:
# ── Cell 6: Parse IEX schedule export & build per-interval rows ────────────────

raw = input_data(folder_paths["input_iex"])

# Rename unnamed columns from the IEX export format
IEX = (
    raw.select([c for c in raw.columns if c != "__UNNAMED__4"])
       .rename({
           "Agent Schedules" : "Agent",
           "__UNNAMED__1"    : "Date",
           "__UNNAMED__2"    : "Start_Shift",
           "__UNNAMED__3"    : "End_Shift",
           "__UNNAMED__5"    : "Scheduled Activity",
           "__UNNAMED__6"    : "Start_Action",
           "__UNNAMED__9"    : "End_Action"
       })
       .with_columns(pl.all().cast(pl.String, strict=False))
)

# Extract "Generation Date" from header rows, then forward/backward fill
IEX = (
    IEX.with_columns([
        pl.when(pl.col("Agent").str.contains("Generation Date: ", literal=True))
          .then(pl.col("Agent").str.extract(r"Generation Date: (.+)", 1))
          .otherwise(None)
          .alias("Generate Date")
    ])
    .with_columns([
        # Fill Generate Date downward so every row knows its export date
        pl.col("Generate Date").fill_null(strategy="backward"),
        pl.col("Agent").fill_null(strategy="forward")
    ])
)

# ── FIX: extract offTable AFTER Generate Date has been filled ─────────────────
# Previously offTable was built before fill_null(backward), so Generate Date
# could still be null for Off-day rows.
offTable = (
    IEX.filter(pl.col("Start_Shift") == "Off")
       .with_columns([
           # If no explicit activity, use the shift state ("Off") as the label
           pl.col("Scheduled Activity").fill_null(pl.col("Start_Shift")),
           pl.lit(None, dtype=pl.String).alias("Start_Shift")
       ])
)

IEX_edit = (
    IEX.filter(
        (pl.col("Start_Shift").fill_null("") != "Off") &
        (pl.col("Date").fill_null("") != "Date") &         # drop column-header rows
        ~(pl.col("Date").is_null() & pl.col("Scheduled Activity").is_null())  # drop blank rows
    )
    .with_columns([
        pl.col("Date").fill_null(strategy="forward"),
        pl.col("Start_Shift").fill_null(strategy="forward"),
        pl.col("End_Shift").fill_null(strategy="forward")
    ])
    .filter(pl.col("Scheduled Activity").is_not_null())
)

combined_df = (
    pl.concat([offTable, IEX_edit], how="diagonal")
    .with_columns([
        pl.col("Export time").cast(pl.String),
        pl.col("Date").cast(pl.String),
        pl.col("Generate Date").cast(pl.String),
        # Extract numeric IEX ID from agent string (e.g. "John Smith (12345)")
        pl.col("Agent").str.extract(r"(\d+)", 1).cast(pl.Int64, strict=False).alias("IEX_ID")
        # NOTE: "Year" and "Month" columns removed — they were dead code
    ])
)

# Keep only the latest Generate Date per agent/date to avoid duplicates
max_generate_date_df = (
    combined_df
    .group_by(["IEX_ID", "Date"])
    .agg(pl.col("Generate Date").max().alias("Max Generate Date"))
)

# Activities that count as productive
productive_activities = ["Open Time", "Extra Hours"]

# Columns to carry forward into intervals_df
target_columns = [
    "Date", "IEX ID", "Agent Email", "First Shift",
    "Scheduled Activity", "Start_Action", "End_Action",
    "PST_Datetime", "Duration", "Work Category"
]
# NOTE: "schedule_flags", "compare_cols", "OracleID" removed — unused dead code

filtered_max_generate_date = (
    combined_df
    .join(max_generate_date_df, on=["IEX_ID", "Date"], how="left")
    .filter(pl.col("Generate Date") == pl.col("Max Generate Date"))
    .with_columns([
        # Flag whether agent has any scheduled activity that day
        pl.when(pl.col("Scheduled Activity").is_not_null().any().over(["Date", "IEX_ID"]))
          .then(1).otherwise(0).cast(pl.Int8, strict=False).alias("Scheduled"),
        pl.col("Export time").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S%.f", strict=False),
        pl.col("Date").str.strip_chars().str.strptime(pl.Date, "%m/%d/%y", strict=False),
        pl.col("Start_Shift").str.strptime(pl.Time, "%I:%M %p", strict=False),
        pl.col("End_Shift").str.strptime(pl.Time, "%I:%M %p", strict=False),
        pl.col("Start_Action").str.strptime(pl.Time, "%I:%M %p", strict=False),
        pl.col("End_Action").str.strptime(pl.Time, "%I:%M %p", strict=False),
        pl.col("IEX_ID").cast(pl.Int64, strict=False)
    ])
    .with_columns([
        # Build shift label e.g. "0800-1700"
        (pl.col("Start_Shift").dt.strftime("%H%M") + "-" + pl.col("End_Shift").dt.strftime("%H%M")).alias("Shift"),
        # Snap Start_Action to the nearest 30-min boundary for interval alignment
        pl.when(pl.col("Start_Action").dt.minute() >= 30)
          .then(pl.col("Start_Action").dt.strftime("%H") + ":30")
          .otherwise(pl.col("Start_Action").dt.strftime("%H") + ":00")
          .alias("VNT")
    ])
    # Construct VNT datetime with timezone, then convert to PST
    .with_columns([
        (pl.col("Date").cast(pl.String) + " " + pl.col("VNT"))
          .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M", strict=False)
          .dt.replace_time_zone("Asia/Ho_Chi_Minh")
          .alias("VNT_Datetime")
    ])
    .with_columns([
        pl.col("VNT_Datetime").dt.convert_time_zone("America/Los_Angeles").alias("PST_Datetime")
    ])
    # Strip timezone info for consistent naive datetime storage
    .with_columns([
        pl.col("VNT_Datetime").dt.replace_time_zone(None),
        pl.col("PST_Datetime").dt.replace_time_zone(None)
    ])
)

# ── Explode each activity into 30-min intervals with partial-minute Duration ───
intervals_df = (
    filtered_max_generate_date
    .with_columns([
        # Fall back to shift times when specific action times are missing
        pl.col("Start_Action").fill_null(pl.col("Start_Shift")),
        pl.col("End_Action").fill_null(pl.col("End_Shift"))
    ])
    .filter(pl.col("Start_Action").is_not_null() & pl.col("End_Action").is_not_null())
    .with_columns([
        (pl.col("Date").cast(pl.String) + " " + pl.col("Start_Action").cast(pl.String))
          .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False)
          .alias("Datetime_Start_Action"),
        (pl.col("Date").cast(pl.String) + " " + pl.col("End_Action").cast(pl.String))
          .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False)
          .alias("Datetime_End_Action")
    ])
    # Handle overnight shifts: if end < start, push end to next day
    .with_columns([
        pl.when(pl.col("Datetime_End_Action") < pl.col("Datetime_Start_Action"))
          .then(pl.col("Datetime_End_Action") + pl.duration(days=1))
          .otherwise(pl.col("Datetime_End_Action"))
          .alias("Datetime_End_Action")
    ])
    # Determine the 30-min interval boundaries that cover this activity
    .with_columns([
        pl.col("Datetime_Start_Action").dt.truncate("30m").alias("Interval_Start_Floor"),
        pl.when(pl.col("Datetime_End_Action") == pl.col("Datetime_End_Action").dt.truncate("30m"))
          .then(pl.col("Datetime_End_Action"))
          .otherwise(pl.col("Datetime_End_Action").dt.truncate("30m") + pl.duration(minutes=30))
          .alias("Interval_End_Ceil")
    ])
    # Generate one timestamp per 30-min slot, then explode into individual rows
    .with_columns([
        pl.datetime_ranges(
            pl.col("Interval_Start_Floor"),
            pl.col("Interval_End_Ceil"),
            interval="30m",
            closed="left"
        ).alias("VNT_Intervals")
    ])
    .explode("VNT_Intervals")
    .with_columns([
        (pl.col("VNT_Intervals") + pl.duration(minutes=30)).alias("Datetime_End_Time_Full"),
        # Actual start within the interval = max(interval_start, activity_start)
        pl.max_horizontal("VNT_Intervals", "Datetime_Start_Action").alias("Datetime_Start_Time")
    ])
    # Actual end within the interval = min(interval_end, activity_end)
    .with_columns([
        pl.min_horizontal("Datetime_End_Time_Full", "Datetime_End_Action").alias("Datetime_End_Time")
    ])
    # Drop zero-duration edge rows
    .filter(pl.col("Datetime_Start_Time") < pl.col("Datetime_End_Time"))
    .with_columns([
        # Duration in hours (partial for boundary intervals)
        ((pl.col("Datetime_End_Time") - pl.col("Datetime_Start_Time")).dt.total_minutes() / 60.0)
          .alias("Duration"),
        # Convert VNT interval timestamp to PST for joining with outage_db
        pl.col("VNT_Intervals")
          .dt.replace_time_zone("Asia/Ho_Chi_Minh")
          .dt.convert_time_zone("America/Los_Angeles")
          .dt.replace_time_zone(None)
          .alias("PST_Intervals"),
        # Classify activity as Productive / Unproductive
        pl.when(pl.col("Scheduled Activity").is_in(productive_activities))
          .then(pl.lit("Productive"))
          .otherwise(pl.lit("Unproductive"))
          .alias("Work Category")
    ])
    .with_columns([
        (pl.col("VNT_Intervals").dt.strftime("%H:%M") + "-" +
         pl.col("Datetime_End_Time_Full").dt.strftime("%H:%M")).alias("VNT_Interval_Range"),
        (pl.col("PST_Intervals").dt.strftime("%H:%M") + "-" +
         (pl.col("PST_Intervals") + pl.duration(minutes=30)).dt.strftime("%H:%M")).alias("PST_Interval_Range")
    ])
    .drop(["Interval_Start_Floor", "Interval_End_Ceil", "Datetime_End_Time_Full", "PST_Datetime"], strict=False)
)

# ── FIX: check available columns AFTER rename, not before ─────────────────────
# Original code checked `intervals_df.columns` (pre-rename) which could drop
# newly renamed columns like "IEX ID", "First Shift", "PST_Datetime".
intervals_df = intervals_df.rename({
    "IEX_ID"      : "IEX ID",
    "Shift"        : "First Shift",
    "PST_Intervals": "PST_Datetime"
})

intervals_df = (
    intervals_df
    # Select only columns that exist in the renamed df
    .select([c for c in target_columns if c in intervals_df.columns])
    .with_columns([
        pl.col("Date").cast(pl.Date, strict=False),
        pl.col("IEX ID").cast(pl.Int64, strict=False)
    ])
    # Enrich with agent metadata (name, LOB, email, supervisor)
    .join(hc_extend_combination, on=["Date", "IEX ID"], how="left")
    .rename({"Email Id": "Agent Email"}, strict=False)
)

# ── FIX: lowercase Agent Email at source to ensure consistent join keys ────────
# Previously only lowercased inline during the join; doing it here keeps the
# column consistent for any future use of intervals_df.
intervals_df = intervals_df.with_columns(
    pl.col("Agent Email").cast(pl.String).str.to_lowercase()
)

# Reorder: key identity columns first, then metrics
priority_cols = ["Date", "IEX ID", "Employee Name", "LOB", "Agent Email", "Supervisor Name"]
intervals_df = intervals_df.select(
    [c for c in priority_cols if c in intervals_df.columns] +
    [c for c in intervals_df.columns if c not in priority_cols]
)

# ── Compare live agent states with their scheduled activities ──────────────────
df_all_details = (
    outage_db
    .filter(pl.col("PST_Datetime") == latest_dt)
    .select(["Agent Name", "Agent Email", "LOB", "Business Location",
             "Connect State", "Duration", "PST_Datetime"])
    .pipe(add_time_cols)
)

df_compare = (
    df_all_details
    .filter(pl.col("Business Location") == "Concentrix (Ho Chi Minh City)")
    .join(
        # Agent Email already lowercased in intervals_df above
        intervals_df.with_columns(
            pl.col("PST_Datetime").cast(pl.Datetime, strict=False)
        ),
        left_on=[
            pl.col("PST_Datetime").cast(pl.Datetime, strict=False),
            pl.col("Agent Email").cast(pl.String).str.to_lowercase()
        ],
        right_on=["PST_Datetime", "Agent Email"],
        how="left"
    )
    .rename({"LOB_right": "LOB_Detail"}, strict=False)
)

display_order = [
    "PST_Date", "PST", "IST", "CLT", "VNT",
    "Agent Name", "Supervisor Name", "First Shift", "LOB_Detail",
    "Duration", "Connect State", "Scheduled Activity",
    "Start_Action", "End_Action", "Work Category"
]
df_compare = df_compare.select([c for c in ["LOB"] + display_order if c in df_compare.columns])

# Split by LOB and active/inactive status for display
df_lodging_active     = df_compare.filter((pl.col("LOB") == "Lodging Chat")     &  pl.col("Connect State").is_in(ACTIVE_STATES)).sort(["Connect State"], nulls_last=True)
df_lodging_inactive   = df_compare.filter((pl.col("LOB") == "Lodging Chat")     & ~pl.col("Connect State").is_in(ACTIVE_STATES)).sort(["Connect State"], nulls_last=True)
df_non_lodging_active = df_compare.filter((pl.col("LOB") == "Non Lodging Chat") &  pl.col("Connect State").is_in(ACTIVE_STATES)).sort(["Connect State"], nulls_last=True)
df_non_lodging_inactive = df_compare.filter((pl.col("LOB") == "Non Lodging Chat") & ~pl.col("Connect State").is_in(ACTIVE_STATES)).sort(["Connect State"], nulls_last=True)

with pl.Config(tbl_rows=-1):
    display(df_lodging_active)
    display(df_lodging_inactive)
    display(df_non_lodging_active)
    display(df_non_lodging_inactive)

LOB,PST_Date,PST,IST,CLT,VNT,Agent Name,Supervisor Name,First Shift,LOB_Detail,Duration,Connect State,Scheduled Activity,Start_Action,End_Action,Work Category
str,date,str,str,str,str,str,str,str,str,time,str,str,time,time,str
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Ho, Quy Minh""","""Mia Minh Le""","""0700-1600""","""Lodging""",00:00:22,"""AVAILABLECHAT""","""Open Time""",08:30:00,12:00:00,"""Productive"""
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Le, Thuy Trieu""","""Ann""","""0600-1500""","""Lodging""",00:00:49,"""AVAILABLECHAT""","""Open Time""",10:30:00,13:15:00,"""Productive"""
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Ngo, Ha Trang""","""Mia Minh Le""","""0600-1500""","""Lodging""",00:00:47,"""AVAILABLECHAT""","""Open Time""",10:30:00,13:30:00,"""Productive"""
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Ha Tu Uyen""","""Tran Thao Uyen""","""0600-1500""","""Lodging""",00:05:08,"""AVAILABLECHAT""","""Open Time""",07:30:00,13:00:00,"""Productive"""
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Khac Bao Thai""","""Chau Thien Kim""","""0700-1600""","""Lodging""",00:08:09,"""AVAILABLECHAT""","""Open Time""",10:15:00,12:15:00,"""Productive"""
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Ngoc Diem Quynh""","""Mia Minh Le""","""0600-1500""","""Lodging""",00:00:05,"""AVAILABLECHAT""","""Open Time""",10:30:00,13:30:00,"""Productive"""
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Tran, Duc Tin""","""Chau Thien Kim""","""0600-1500""","""Lodging""",00:00:40,"""AVAILABLECHAT""","""Lunch""",10:15:00,11:15:00,"""Unproductive"""
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Tran, Duc Tin""","""Chau Thien Kim""","""0600-1500""","""Lodging""",00:00:40,"""AVAILABLECHAT""","""Open Time""",11:15:00,13:00:00,"""Productive"""
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Tran, Thao Uyen""",null,null,null,00:05:14,"""AVAILABLECHAT""",null,null,null,null


LOB,PST_Date,PST,IST,CLT,VNT,Agent Name,Supervisor Name,First Shift,LOB_Detail,Duration,Connect State,Scheduled Activity,Start_Action,End_Action,Work Category
str,date,str,str,str,str,str,str,str,str,time,str,str,time,time,str
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Hai Nam""","""Chau Thien Kim""","""0600-1500""","""Lodging""",00:16:56,"""BREAK""","""Open Time""",10:30:00,13:30:00,"""Productive"""
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Thi Tuyet Mai""","""Tran Thao Uyen""","""0700-1600""","""Lodging""",00:04:23,"""BREAK""","""Open Time""",09:30:00,12:00:00,"""Productive"""
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Pham, Thi Tuyet Nhi""",null,null,null,18:51:41,"""LOGIN""",null,null,null,null
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Bui, Cong Danh""","""Tran Thao Uyen""","""0600-1500""","""Lodging""",00:16:54,"""LUNCH""","""Lunch""",11:00:00,12:00:00,"""Unproductive"""
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Hoang, Le Vy""","""Ann""","""0600-1500""","""Lodging""",00:17:10,"""LUNCH""","""Open Time""",09:15:00,11:15:00,"""Productive"""
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Hoang, Le Vy""","""Ann""","""0600-1500""","""Lodging""",00:17:10,"""LUNCH""","""Lunch""",11:15:00,12:15:00,"""Unproductive"""
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Quang Hien""","""Mia Minh Le""","""0600-1500""","""Lodging""",00:02:09,"""LUNCH""","""Open Time""",09:15:00,11:15:00,"""Productive"""
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Quang Hien""","""Mia Minh Le""","""0600-1500""","""Lodging""",00:02:09,"""LUNCH""","""Lunch""",11:15:00,12:15:00,"""Unproductive"""
"""Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Thi Kim Ngan""","""Chau Thien Kim""","""0600-1500""","""Lodging""",00:18:49,"""LUNCH""","""Open Time""",09:15:00,11:15:00,"""Productive"""


LOB,PST_Date,PST,IST,CLT,VNT,Agent Name,Supervisor Name,First Shift,LOB_Detail,Duration,Connect State,Scheduled Activity,Start_Action,End_Action,Work Category
str,date,str,str,str,str,str,str,str,str,time,str,str,time,time,str
"""Non Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Dinh, Thi Ngoc Han""","""Mia Minh Le""","""0600-1500""","""Non_Lodging""",00:06:37,"""AVAILABLECHAT""","""Open Time""",10:45:00,13:30:00,"""Productive"""
"""Non Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Quynh Tuyet Kha""","""Mia Minh Le""","""0600-1500""","""Non_Lodging""",00:13:53,"""AVAILABLECHAT""","""Open Time""",10:15:00,13:30:00,"""Productive"""
"""Non Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Vu, Minh Nghia""","""Mia Minh Le""","""0600-1500""","""Non_Lodging""",00:12:07,"""AVAILABLECHAT""","""Open Time""",10:45:00,13:15:00,"""Productive"""


LOB,PST_Date,PST,IST,CLT,VNT,Agent Name,Supervisor Name,First Shift,LOB_Detail,Duration,Connect State,Scheduled Activity,Start_Action,End_Action,Work Category
str,date,str,str,str,str,str,str,str,str,time,str,str,time,time,str
"""Non Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Le Tam Lien""","""Tran Thao Uyen""","""0700-1600""","""Non_Lodging""",00:28:43,"""BREAK""","""Break""",11:00:00,11:15:00,"""Unproductive"""
"""Non Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Le Tam Lien""","""Tran Thao Uyen""","""0700-1600""","""Non_Lodging""",00:28:43,"""BREAK""","""Open Time""",11:15:00,13:00:00,"""Productive"""
"""Non Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Lai, Ngoc Ai Linh""","""Mia Minh Le""","""0700-1600""","""Non_Lodging""",00:05:17,"""LUNCH""","""Open Time""",09:15:00,11:30:00,"""Productive"""
"""Non Lodging Chat""",2026-05-13,"""21:00""","""09:30""","""07:00""","""11:00""","""Trinh, Hoang Nam""","""Mia Minh Le""","""0700-1600""","""Non_Lodging""",00:27:29,"""LUNCH""","""Open Time""",11:00:00,13:45:00,"""Productive"""


In [7]:
# ── Cell 7: Schedule summary — headcount grid by VN interval ──────────────────

TARGET_DATE = "2026-05-13"
START_TIME  = "06:00"
END_TIME    = "18:00"

# Aggregate IEX intervals into headcount per activity group per 30-min slot
df_summary = (
    intervals_df
    .with_columns([
        pl.col("PST_Datetime")
          .dt.replace_time_zone("America/Los_Angeles")
          .dt.convert_time_zone("Asia/Ho_Chi_Minh")
          .dt.replace_time_zone(None)
          .alias("VN_Datetime")
    ])
    .with_columns([
        pl.col("VN_Datetime").dt.date().alias("VN_Date"),
        pl.col("VN_Datetime").dt.strftime("%H:%M").alias("VN_Interval"),
        pl.col("PST_Datetime").dt.date().alias("PST_Date"),
        pl.col("PST_Datetime").dt.strftime("%H:%M").alias("PST_Interval"),

        # Group scheduled activities into display categories
        pl.when(pl.col("Scheduled Activity").is_in(["Open Time", "Extra Hours"])).then(pl.lit("Available"))
          .when(pl.col("Scheduled Activity") == "Break").then(pl.lit("Break"))
          .when(pl.col("Scheduled Activity") == "Lunch").then(pl.lit("Lunch"))
          .when(pl.col("Scheduled Activity").str.contains("(?i)Training")).then(pl.lit("Training"))
          .when(pl.col("Scheduled Activity").is_in(["PTO", "Paid Leave"])).then(pl.lit("PL"))
          .when(pl.col("Scheduled Activity") == "No Call/No Show").then(pl.lit("NCNS"))
          .otherwise(pl.lit("Other"))
          .alias("Activity_Group"),

        # Duration * 2 converts fractional-hour duration to headcount equivalent
        (pl.col("Duration") * 2).alias("Headcount")
    ])
    .pivot(
        values="Headcount",
        index=["VN_Date", "PST_Date", "VN_Interval", "PST_Interval", "LOB"],
        columns="Activity_Group",
        aggregate_function="sum"
    )
    .fill_null(0)
)

# Ensure all expected activity columns exist even if no data for that category
for col in ["Available", "Break", "Lunch", "Training", "PL", "NCNS"]:
    if col not in df_summary.columns:
        df_summary = df_summary.with_columns(pl.lit(0).alias(col))

# Build the full time grid for TARGET_DATE (zero-padded HH:MM ensures correct string sort)
custom_intervals = [
    i for i in [f"{h:02d}:{m:02d}" for h in range(24) for m in (0, 30)]
    if START_TIME <= i <= END_TIME
]

grid_df = (
    pl.DataFrame({
        "VN_Date": [TARGET_DATE] * 4,
        "LOB"    : ["Lodging", "Lodging_Nesting", "Non_Lodging", "Non_Lodging_Nesting"]
    })
    .with_columns(pl.col("VN_Date").str.strptime(pl.Date, "%Y-%m-%d"))
    .join(pl.DataFrame({"VN_Interval": custom_intervals}), how="cross")
    .with_columns([
        (pl.col("VN_Date").cast(pl.String) + " " + pl.col("VN_Interval"))
          .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M")
          .dt.replace_time_zone("Asia/Ho_Chi_Minh")
          .alias("VN_Datetime")
    ])
    .with_columns([
        pl.col("VN_Datetime").dt.convert_time_zone("America/Los_Angeles").alias("PST_Datetime")
    ])
    .with_columns([
        pl.col("PST_Datetime").dt.date().alias("PST_Date"),
        pl.col("PST_Datetime").dt.strftime("%H:%M").alias("PST_Interval")
    ])
    .select(["VN_Date", "PST_Date", "VN_Interval", "PST_Interval", "LOB"])
)

# Left-join actual data onto the full grid so every interval appears even if empty
df_summary_full = (
    grid_df
    .join(
        df_summary.select(["VN_Date", "VN_Interval", "LOB",
                           "Available", "Break", "Lunch", "Training", "PL", "NCNS"]),
        on=["VN_Date", "VN_Interval", "LOB"],
        how="left"
    )
    .fill_null(0)
    .sort(["VN_Date", "LOB", "VN_Interval"])
)

num_cols  = ["Available", "Break", "Lunch", "Training", "PL", "NCNS"]
date_cols = ["VN_Date", "PST_Date"]
time_keys = ["VN_Date", "PST_Date", "VN_Interval", "PST_Interval"]


def aggregate_lob(df, lob_list):
    """Sum headcount across LOB variants (main + nesting) for each interval."""
    return (
        df.filter(pl.col("LOB").is_in(lob_list))
          .group_by(time_keys)
          .agg([pl.col(c).sum() for c in num_cols])
          .sort("VN_Interval")   # safe: zero-padded HH:MM strings sort correctly
    )

df_LG_final = aggregate_lob(df_summary_full, ["Lodging", "Lodging_Nesting"])
df_NL_final = aggregate_lob(df_summary_full, ["Non_Lodging", "Non_Lodging_Nesting"])


def get_styled_html(df, title):
    """Convert a Polars DataFrame to a colour-gradient HTML table."""
    pdf = df.to_pandas()

    # FIX: convert date columns to strings BEFORE calling .style to avoid
    # strftime errors caused by mixed scalar types returned by pandas.
    for c in date_cols:
        if c in pdf.columns:
            pdf[c] = pdf[c].apply(
                lambda x: x.strftime("%d/%m") if hasattr(x, "strftime") else str(x)
            )

    fmt_final = {c: "{:.2f}" for c in num_cols}

    return (
        pdf.style
           .hide(axis="index")
           .set_table_attributes('style="margin-right: 20px; text-align: center; border-collapse: collapse;"')
           .set_caption(f"<b style='font-size: 16px; color: #333;'>{title}</b>")
           .format(fmt_final)
           # Available/productivity columns: more = better → green high end
           .background_gradient(cmap="RdYlGn",   subset=["Available", "Break", "Lunch", "Training"])
           # Absence columns: more = worse → red high end (reversed palette)
           .background_gradient(cmap="RdYlGn_r", subset=["PL", "NCNS"])
           .to_html()
    )

html_output = f"""
<div style="display: flex; align-items: flex-start; overflow-x: auto; padding: 10px;">
    {get_styled_html(df_LG_final, "Lodging")}
    {get_styled_html(df_NL_final, "Non-Lodging")}
</div>
"""

display(HTML(html_output))

C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_24252\2313862134.py:36: DeprecationWarning: The argument `columns` for `DataFrame.pivot` is deprecated. It has been renamed to `on`.
  .pivot(


VN_Date,PST_Date,VN_Interval,PST_Interval,Available,Break,Lunch,Training,PL,NCNS
13/05,12/05,06:00,16:00,26.00,0.00,0.00,5.00,1.00,2.00
13/05,12/05,06:30,16:30,26.00,0.00,0.00,5.00,1.00,2.00
13/05,12/05,07:00,17:00,23.00,6.00,0.00,5.00,2.00,2.00
13/05,12/05,07:30,17:30,26.00,3.00,0.00,5.00,2.00,2.00
13/05,12/05,08:00,18:00,26.50,4.50,0.00,3.00,2.00,2.00
13/05,12/05,08:30,18:30,31.00,0.00,0.00,3.00,2.00,2.00
13/05,12/05,09:00,19:00,30.00,1.00,3.00,0.00,2.00,2.00
13/05,12/05,09:30,19:30,24.83,1.00,8.00,0.17,2.00,2.00
13/05,12/05,10:00,20:00,18.00,1.50,15.00,1.00,2.00,2.00
13/05,12/05,10:30,20:30,23.00,0.00,12.00,1.00,2.00,2.00


In [8]:
# ── Cell 8: HTML dashboard — summary + inactive detail tables ──────────────────

# Reverse lookup: short code → full city name for display
SITE_FULLNAME = {"PUN": "Pune", "HCM": "Ho Chi Minh", "KOL": "Kolkata", "CAI": "Cairo"}

df_merge = (
    df_active
    .join(
        df_inactive.select(["LOB", "Site", "Heads Inactive"]),
        on=["LOB", "Site"],
        how="left"
    )
    .with_columns(pl.col("Site").replace(SITE_FULLNAME))
    .sort(["LOB", "Site"])
)

pdf            = df_merge.to_pandas()
lodging_detail     = df_lodging.to_pandas()
non_lodging_detail = df_non_lodging.to_pandas()


def build_summary_table(lob_name):
    """Build an HTML summary table with rowspan on shared time/date columns."""
    sub = pdf[pdf["LOB"] == lob_name]

    html = f"""
    <div style="width:49%">
    <h3 style="text-align:center">{lob_name}</h3>
    <table border="1" style="width:100%;border-collapse:collapse;text-align:center;font-family:Arial;font-size:12px">
    <tr style="background:#222;color:white">
      <th>PST Date</th><th>PST</th><th>IST</th><th>CLT</th><th>VNT</th>
      <th>Site</th><th>Total OU Req</th><th>OU Req by Site</th>
      <th>Heads Active</th><th>Heads Inactive</th>
      <th>Missing Heads</th><th>Heads Active / OU Req by Site</th>
    </tr>
    """

    for i, (_, r) in enumerate(sub.iterrows()):
        html += "<tr>"

        # Merge time/date cells across all site rows (rowspan)
        if i == 0:
            rowspan = len(sub)
            html += f"""
              <td rowspan="{rowspan}" style="vertical-align:middle">{r['PST_Date'].strftime('%Y-%m-%d')}</td>
              <td rowspan="{rowspan}" style="vertical-align:middle">{r['PST']}</td>
              <td rowspan="{rowspan}" style="vertical-align:middle">{r['IST']}</td>
              <td rowspan="{rowspan}" style="vertical-align:middle">{r['CLT']}</td>
              <td rowspan="{rowspan}" style="vertical-align:middle">{r['VNT']}</td>
            """

        # Colour Missing Heads: red = understaffed, green = overstaffed
        missing_color = (
            '#d32f2f' if r['Missing Heads'] < 0 else
            '#2e7d32' if r['Missing Heads'] > 0 else
            'transparent'
        )
        html += f"""
          <td><b>{r['Site']}</b></td>
          <td>{r['Total OU Req']}</td>
          <td>{r['OU Req by Site']}</td>
          <td>{r['Heads Active']}</td>
          <td>{r['Heads Inactive']}</td>
          <td style="background:{missing_color};color:white;font-weight:bold">{r['Missing Heads']}</td>
          <td>{r['Heads Active / OU Req by Site']}</td>
        </tr>
        """

    html += "</table></div>"
    return html


def build_detail_table(df, title):
    """Build an HTML detail table listing inactive agents with rowspan on time columns."""
    html = f"""
    <div style="width:49%">
    <h3 style="text-align:center">{title} Details</h3>
    <table border="1" style="width:100%;border-collapse:collapse;text-align:center;font-family:Arial;font-size:12px">
    <tr style="background:#222;color:white">
      <th>PST Date</th><th>PST</th><th>IST</th><th>CLT</th><th>VNT</th>
      <th>Agent Name</th><th>Agent Email</th><th>Business Location</th>
      <th>Connect State</th><th>Duration</th>
    </tr>
    """

    rowspan = len(df)

    for i, (_, r) in enumerate(df.iterrows()):
        html += "<tr>"

        if i == 0:
            html += f"""
              <td rowspan="{rowspan}" style="vertical-align:middle">{r['PST_Date'].strftime('%Y-%m-%d')}</td>
              <td rowspan="{rowspan}" style="vertical-align:middle">{r['PST']}</td>
              <td rowspan="{rowspan}" style="vertical-align:middle">{r['IST']}</td>
              <td rowspan="{rowspan}" style="vertical-align:middle">{r['CLT']}</td>
              <td rowspan="{rowspan}" style="vertical-align:middle">{r['VNT']}</td>
            """

        html += f"""
          <td>{r['Agent Name']}</td>
          <td>{r['Agent Email']}</td>
          <td>{r['Business Location']}</td>
          <td>{r['Connect State']}</td>
          <td>{r['Duration']}</td>
        </tr>
        """

    html += "</table></div>"
    return html


html = f"""
<style>
table {{ border-collapse:collapse; text-align:center; font-family:Arial; font-size:13px; table-layout:auto; }}
td, th {{ padding:6px 10px; white-space:nowrap; }}
th {{ background:#222; color:white; }}
</style>

<div style="margin-bottom:25px">{build_summary_table("Lodging Chat")}</div>
<div style="margin-bottom:40px">{build_summary_table("Non Lodging Chat")}</div>
<div style="margin-bottom:25px">{build_detail_table(lodging_detail, "Lodging Chat")}</div>
<div>{build_detail_table(non_lodging_detail, "Non Lodging Chat")}</div>
"""

display(HTML(html))